Here, we download the raw data from [McCauley 2024](https://doi.org/10.1016/j.cels.2024.02.005). 

Single-cell datasets include GEO: GSE246368.

In [1]:
import os
import zipfile

import scanpy as sc
import pandas as pd

import pooch

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

# Download Data

In [3]:
file_path = os.path.join(data_path, "raw")

Cell cycle data (as in [scanpy tutorial](https://scanpy.readthedocs.io/en/1.11.x/how-to/cell-cycle.html)):

In [4]:
p_zip = pooch.retrieve(
    "https://www.dropbox.com/s/3dby3bjsaf5arrw/cell_cycle_vignette_files.zip?dl=1",
    known_hash="sha256:6557fe2a2761d1550d41edf61b26c6d5ec9b42b4933d56c3161164a595f145ab",
    path=file_path
)
with zipfile.ZipFile(p_zip, "r") as f_zip:
#     f_csv = zipfile.Path(f_zip, "nestorawa_forcellcycle_expressionMatrix.txt").open()
#     adata = sc.read_csv(f_csv, delimiter="\t").T
    cell_cycle_genes = zipfile.Path(f_zip, "regev_lab_cell_cycle_genes.txt").read_text().splitlines()

Download the HGNC database (10/22/25), needed for filtering the AnnData object for protein-coding genes:

In [5]:
fn = "HGNC_db.txt"

if not os.path.isfile(os.path.join(file_path, fn)):
    file_link = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/locus_types/gene_with_protein_product.txt"
    fn_0 = "gene_with_protein_product.txt"

    cmd = f"""
    wget -O "{file_path}/{fn_0}" "{file_link}" && \
    mv "{file_path}/{fn_0}" "{file_path}/{fn}"
    """

    os.system(cmd)

Download raw counts of GSE246368 from [Zenodo](https://zenodo.org/records/10602177). This is the same as 'GSE246368_all_data.h5ad.gz' from [GSE246368](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE246368) but with more information (e.g. UMAP). 

Since the data associated with GSE246441 (additional donors for control, OSM, and IFN) was only used by the paper for Fig 1 abundance plots (according to where it was used in the [Github](https://github.com/AllonKleinLab/paper-data/tree/master/McCauley_Kukreja_2024/notebooks)), we will disregard it.

In [6]:
fn = author + "_GSE246368.h5ad"

# GEO version
# # wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE246nnn/GSE246368/suppl/GSE246368%5Fall%5Fdata.h5ad.gz
# # gunzip GSE246368_all_data.h5ad.gz
# # mv GSE246368_all_data.h5ad McCauley_GSE246368.h5ad

# if not os.path.isfile(os.path.join(file_path, fn)):
#     file_link = 'https://zenodo.org/records/10602177/files/all_indrops_data.h5ad.zip?download=1'
#     fn_0 = "all_indrops_data.h5ad.zip?download=1"
#     fn_1 = 'all_indrops_data.h5ad '

#     cmd = f"""
#     wget -O "{file_path}/{fn_0}.gz" "{file_link}" && \
#     gunzip -f "{file_path}/{fn_0}.gz" && \
#     mv "{file_path}/{fn_1}" "{file_path}/{fn}"
#     """

#     os.system(cmd)


# Zenodo version
# wget https://zenodo.org/records/10602177/files/all_indrops_data.h5ad.zip?download=1
# unzip 'all_indrops_data.h5ad.zip?download=1'
# rm 'all_indrops_data.h5ad.zip?download=1'
# mv all_indrops_data.h5ad McCauley_GSE246368.h5ad

if not os.path.isfile(os.path.join(file_path, fn)):
    file_link = "https://zenodo.org/records/10602177/files/all_indrops_data.h5ad.zip?download=1"
    fn_0 = "all_indrops_data.h5ad.zip?download=1"

    cmd = f"""
    wget -O "{file_path}/{fn_0}" "{file_link}" && \
    unzip "{file_path}/{fn_0}" -d "{file_path}" && \
    rm "{file_path}/{fn_0}" && \
    mv "{file_path}/all_indrops_data.h5ad" "{file_path}/{fn}"
    """

    os.system(cmd)

In [8]:
file_path = os.path.join(data_path, "raw")
fn = author + "_GSE246368.h5ad"
adata = sc.read_h5ad(os.path.join(file_path, fn))


# QC

In [9]:
# filter out doublets and unassigned cells
adata = adata[~adata.obs.assigned_donor.isin(['doublet', 'Unassigned'])] 

# retain only feaetures that are protein coding genes
hgnc_genes = pd.read_csv(os.path.join(data_path, 'raw', 'HGNC_db.txt'), 
                        sep = '\t', low_memory=False).symbol.tolist()
gene_mask = [True if vn in hgnc_genes else False for vn in adata.var_names]
adata = adata[:, gene_mask]

# basic QC -- same thresholds as in original paper (https://github.com/AllonKleinLab/paper-data/blob/master/McCauley_Kukreja_2024/notebooks/fig1_2_analyses/control_neighbors_analyses_fig1.ipynb)
sc.pp.filter_genes(adata, min_counts=6)
sc.pp.filter_genes(adata, min_cells=3)


/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:291: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_counts"] = number


# Metadata Formatting

Column Names:

In [10]:
cat_col = 'cell_type'
pert_col = 'ligand'

adata.obs.rename(
    columns = {
        'cell line': 'donor_ID', 
        'condition': pert_col, 
        'cell_states': cat_col
    }, inplace = True
)

Ligand column mapping to match Omnipath DB:

In [11]:
# map for coherence with omnipath
pert_map = {
    # adjusted annotation to agree with omnipath
    
    # missing in Omnipath and needs renaming
    'None': 'CTRL',
    'ActA': 'INHBA', 
    'Adipo': 'ADIPOQ', 
    'CHIR': 'CHIR99021', # small molecule - will need to add to omnipath
    'IFNA': 'IFNA2', # alpha 2a from star methods
    'Leptin': 'LEP', 
    
    # needs alterations in Omnipath
    'TNFA': 'TNFA', # omnipath only has TNF, will need a more specific network for this

    
    # doesn't need adjusting
    'BMP4': 'BMP4', 
    'EGF': 'EGF', 
    'FGF10': 'FGF10', 
    'FGF2': 'FGF2', # incorrect IDENTIFIER in star methods, but will assume this is an accurate mapping
    'HGF': 'HGF', 
    'IFNG': 'IFNG', 
    'IGF1': 'IGF1', # did not see in star methods, but will assume this is an accurate mapping
    'IL13': 'IL13', 
    'IL17A': 'IL17A', # incorrect IDENTIFIER in star methods, but will assume this is an accurate mapping
    'OSM': 'OSM',
    'TGFB1': 'TGFB1' # did not see in start methods, but will assume this is an accurate mapping

}
adata.obs[pert_col] = adata.obs[pert_col].map(pert_map)


Improved labeling of subclusters:

A fraction of the cells appeared transitional between basal and secretory (Figures 2B and S3B–S3D), and between secretory and multiciliated (Figures 2B and S3E–S3G). 
- basal --> secretory transition lies on both goblet and club cells, so these each of their own mature secretory and transitional secretory columns 
- secretory --> multiciliates has a unique mapping

"We grouped these cells with mature secretory or multiciliated cells and confirmed that doing so preserved the fold-changes in cell-type abundances upon treatment that are analyzed below (Figure S3I)." --> *may* have distinct transcriptional responses, but we will proceed with the higher-order label for now. 

In [12]:
print('Original mapping: ')
print()
ct_resolution_map = {}
for ct in adata.obs['cell_type'].unique():
    mask = (adata.obs['cell_type'] == ct)
    ct_resolution_map[ct] = sorted(adata.obs.loc[mask, 'secretory_mcc_subsets'].unique())

for ct, ct_sub in ct_resolution_map.items():
    ct_sub_ = ct_sub[0] if len(ct_sub) == 1 else ct_sub
    print('{}: {}'.format(ct, ct_sub_))

Original mapping: 

Club: ['Secretory-mature', 'Secretory-transitional']
Basal: Basal
Goblet: ['Secretory-mature', 'Secretory-transitional']
Multiciliated: ['Multiciliated-mature', 'Multiciliated-transitional']
Tuft: Tuft
Ionocyte: Ionocyte
PNEC: PNEC
Indeterminate: Indeterminate


In [13]:
mask = adata.obs['cell_type'].isin(['Club', 'Goblet'])
current_cats = adata.obs['secretory_mcc_subsets'].cat.categories

new_vals = (
    adata.obs.loc[mask, 'cell_type'].astype(str)
    + ': '
    + adata.obs.loc[mask, 'secretory_mcc_subsets'].astype(str)
).unique()

adata.obs['secretory_mcc_subsets'] = adata.obs['secretory_mcc_subsets'].cat.add_categories(new_vals)

adata.obs.loc[mask, 'secretory_mcc_subsets'] = (
    adata.obs.loc[mask, 'cell_type'].astype(str)
    + ': '
    + adata.obs.loc[mask, 'secretory_mcc_subsets'].astype(str)
)

adata.obs['secretory_mcc_subsets'] = (
    adata.obs['secretory_mcc_subsets'].cat.remove_unused_categories()
)

print('Specified mapping: ')
print()
ct_resolution_map = {}
for ct in adata.obs['cell_type'].unique():
    mask = (adata.obs['cell_type'] == ct)
    ct_resolution_map[ct] = sorted(adata.obs.loc[mask, 'secretory_mcc_subsets'].unique())
for ct, ct_sub in ct_resolution_map.items():
    ct_sub_ = ct_sub[0] if len(ct_sub) == 1 else ct_sub
    print('{}: {}'.format(ct, ct_sub_))
    


Specified mapping: 

Club: ['Club: Secretory-mature', 'Club: Secretory-transitional']
Basal: Basal
Goblet: ['Goblet: Secretory-mature', 'Goblet: Secretory-transitional']
Multiciliated: ['Multiciliated-mature', 'Multiciliated-transitional']
Tuft: Tuft
Ionocyte: Ionocyte
PNEC: PNEC
Indeterminate: Indeterminate


In [14]:
adata.obs['condition'] = adata.obs[cat_col].astype(str) + '^' + adata.obs[pert_col].astype(str)

In [15]:
adata.obs[cat_col].value_counts(normalize = True)*100

cell_type
Basal            49.773922
Club             32.145289
Multiciliated    15.198976
Goblet            1.758233
Ionocyte          0.543404
Tuft              0.329584
Indeterminate     0.140277
PNEC              0.110315
Name: proportion, dtype: float64

Drop unlabelled cells and rare cell types.

The paper says some of the rare cell types are lost in some conditions.

- "Whether these pathways only suppress the differentiation of secretory and multiciliated cells, or also lead to the loss of rare cell types (ionocyte, PNEC, and tuft cells), was until now not known. We found that all the above-mentioned conditions led to the depletion of rare luminal cell types, and with CHIR and BMP4 depleting the rare cell types to a much higher extent than secretory and multiciliated cells". They also show the fewest DE genes.
- While Fig 3D indicates that Tuf and Ionocyte do get some perturbation signal, given the above and the fact that when combined with perturbaiton, there are vary few cell types in that condition, we will eliminate all the rare cell types. 




In [16]:
unlabelled_cells = ['Indeterminate']
rare_cells = ['PNEC', 'Tuft', 'Ionocyte']

mask = ~adata.obs[cat_col].isin(unlabelled_cells + rare_cells)
adata = adata[mask].copy()

for col in [cat_col, 'secretory_mcc_subsets']:
    adata.obs[col] = (
        adata.obs[col].cat.remove_unused_categories()
    )
    
adata.obs[cat_col].value_counts(normalize = True)*100

cell_type
Basal            50.339527
Club             32.510571
Multiciliated    15.371689
Goblet            1.778212
Name: proportion, dtype: float64

Finally, we also exclude cell type - perturbation combinations with fewer than 50 cells:

In [17]:
count_thresh = 30
low_count_conditions = adata.obs['condition'].value_counts()[lambda x: x < count_thresh].index
print('{} of {} conditions have fewer than 100 cells and will be excluded'.format(
    len(low_count_conditions), adata.obs['condition'].nunique()))

mask = ~(adata.obs['condition'].isin(low_count_conditions))
adata = adata[mask, :].copy()

7 of 68 conditions have fewer than 100 cells and will be excluded


Let's view the final condition frequencies:

In [18]:
freq_matrix = pd.crosstab(adata.obs[cat_col], adata.obs[pert_col])
freq_matrix.loc['Total'] = freq_matrix.sum(axis=0)
freq_matrix['Total'] = freq_matrix.sum(axis=1)

# freq_matrix_norm = 100 * (freq_matrix / freq_matrix.loc['Total', 'Total'])
# freq_matrix_norm.T

freq_matrix.T


cell_type,Basal,Club,Goblet,Multiciliated,Total
ligand,,,,,
INHBA,1679,2068,0,990,4737
ADIPOQ,1932,1713,0,440,4085
BMP4,2294,858,0,555,3707
CHIR99021,1071,1022,0,329,2422
EGF,1773,1717,39,510,4039
FGF2,1142,590,0,523,2255
FGF10,2683,1119,0,471,4273
HGF,1227,1387,0,610,3224
IFNA2,3179,525,57,596,4357


# Normalization/HVG

In [19]:
# normalize
adata.layers["counts"] = adata.X.copy() # store the raw data
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)

# hvgs
sc.pp.highly_variable_genes(adata, n_top_genes=5000, batch_key=None, flavor = 'seurat')


# cell cycle scoring
# scanpy tutorial(https://scanpy.readthedocs.io/en/1.11.x/how-to/cell-cycle.html)
s_genes = [x for x in cell_cycle_genes[:43] if x in adata.var_names]
g2m_genes = [x for x in cell_cycle_genes[43:] if x in adata.var_names]
cell_cycle_genes = [*s_genes, *g2m_genes]
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)


Some final formatting:

In [20]:
adata.write_h5ad(os.path.join(data_path, 'processed', author + '_normalized_counts.h5ad'))